In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
%matplotlib inline

# Загрузка данных
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)
!unzip -q hw_light.zip

# Подготовка данных
base_dir = '/content/hw_light'
x_data = []
y_data = []
img_height = 20
img_width = 20

for patch in os.listdir(base_dir):
    for img in os.listdir(base_dir + '/' + patch):
        img_path = base_dir + '/' + patch + '/' + img
        image = Image.open(img_path).convert('L')
        image = image.resize((img_height, img_width))
        img_array = np.array(image, dtype=np.float32) / 255.0
        x_data.append(img_array)

        if patch == '0':
            y_data.append(0)  # треугольник
        elif patch == '3':
            y_data.append(1)  # круг
        else:
            y_data.append(2)  # квадрат

x_data = np.array(x_data).reshape(-1, 1, 20, 20)
y_data = np.array(y_data)

print('Размер массива x_data', x_data.shape)
print('Размер массива y_data', y_data.shape)

# Разделение на train и validation
x_train, x_val, y_train, y_val = train_test_split(x_data, y_data, test_size=0.2, random_state=42)

# Преобразование в тензоры
x_train_tensor = torch.FloatTensor(x_train)
y_train_tensor = torch.LongTensor(y_train)
x_val_tensor = torch.FloatTensor(x_val)
y_val_tensor = torch.LongTensor(y_val)

In [ ]:
def train_model(neurons, activation, batch_size, epochs=10):
    # Определение модели
    class ShapeModel(nn.Module):
        def __init__(self):
            super(ShapeModel, self).__init__()
            self.flatten = nn.Flatten()
            if activation == 'relu':
                self.fc1 = nn.Linear(400, neurons)
                self.activation = nn.ReLU()
            else:  # linear
                self.fc1 = nn.Linear(400, neurons)
                self.activation = nn.Identity()
            self.fc2 = nn.Linear(neurons, 3)

        def forward(self, x):
            x = self.flatten(x)
            x = self.activation(self.fc1(x))
            x = self.fc2(x)
            return x

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ShapeModel().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # DataLoaders
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Обучение
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Оценка на всех данных
    model.eval()
    all_images = torch.FloatTensor(x_data).to(device)
    with torch.no_grad():
        outputs = model(all_images)
        _, predicted = torch.max(outputs.data, 1)
        accuracy = (predicted.cpu().numpy() == y_data).mean()

    return accuracy

In [ ]:
results = []

# Параметры для экспериментов
neurons_list = [10, 100, 5000]
activations = ['relu', 'linear']
batch_sizes = [10, 100, 1000]

for neurons in neurons_list:
    for activation in activations:
        for batch_size in batch_sizes:
            print(f"Training: neurons={neurons}, activation={activation}, batch_size={batch_size}")
            acc = train_model(neurons, activation, batch_size)
            results.append([neurons, activation, batch_size, acc])
            print(f"  Accuracy: {acc:.4f}")

In [ ]:
df = pd.DataFrame(results, columns=["Нейроны", "Активация", "Batch size", "Точность"])
print(df)

In [ ]:

# Сохраняем веса последней обученной модели
class FinalShapeModel(nn.Module):
    def __init__(self, neurons=5000, activation='linear'):
        super(FinalShapeModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(400, neurons)
        self.activation = nn.ReLU() if activation == 'relu' else nn.Identity()
        self.fc2 = nn.Linear(neurons, 3)

    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.fc2(x)
        return x

final_model = FinalShapeModel()
# Обучаем финальную модель (можно с лучшими параметрами)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
final_model.to(device)

# Быстрое обучение финальной модели
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=0.001)

for epoch in range(10):
    final_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Сохранение весов
torch.save(final_model.state_dict(), 'lite.pth')
print("✓ Веса сохранены в lite.pth")

# Скачивание (опционально)
from google.colab import files
files.download('lite.pth')